# SEC filing sentiment and equity returns

**Authors: Jeremy Faubert and Matthias Baker.** This research pipeline compares baseline, expanded and industry-matched sentiment portfolios using precomputed Loughran-McDonald filing summaries.

**Research status:** the original profitability and out-of-sample conclusions have been withdrawn. This public implementation does not include a validated historical link input or establish complete investment performance. Full-period statistics require validated identities and resolved held payoffs. See README.md and docs/CORRECTIONS.md.

Default `legacy_diagnostic` mode uses a frozen current ticker map solely to inspect the earlier cohort. It does not validate historical identities. The alternative `verified_links` mode requires an independently reviewed dated input; the name is an input-mode label, not automatic certification.

Formation January 1994-December 2023 corresponds to nominal holding February 1994-January 2024. Each sleeve can use at most one unit per capital unit (up to 200% gross). Unallocated capacity contributes zero excess P&L. Trading, borrow and financing costs are excluded.


In [ ]:
import os
from pathlib import Path

# Start Jupyter in the repository root. In Colab, clone the repo and change
# directory to it before running this notebook (see README).
REPO_DIR = Path(os.environ.get('NLP_REPO_DIR', str(Path.cwd())))
CODE_DIR = REPO_DIR
DATA_DIR = Path(os.environ.get('NLP_DATA_DIR', str(REPO_DIR / 'data')))
OUTPUT_DIR = Path(os.environ.get('NLP_OUTPUT_DIR', str(REPO_DIR / 'research_outputs')))

# Optional Colab mount. Configure NLP_DATA_DIR to point to your licensed files.
MOUNT_GOOGLE_DRIVE = False
if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

IDENTIFIER_MODE = 'legacy_diagnostic'
HISTORICAL_LINKS_FILE = DATA_DIR / 'historical_permno_cik_links.csv'
BASELINE_FORMULA = 'published'  # (P-N)/U+1; alternative 'guide'=(P-N)/(U+1)
RUN_PANEL_REGRESSIONS = True   # Descriptive only; dense month effects use several GB RAM.

assert OUTPUT_DIR.resolve() != DATA_DIR.resolve(), 'Keep outputs separate from source data.'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Mode:', IDENTIFIER_MODE, '| Formula:', BASELINE_FORMULA)
print('Review research status before interpreting any output.')


## Load local inputs and the frozen public ticker snapshot

Place the three CSVs listed in data/README.md in DATA_DIR. They are not bundled or redistributed. Any provenance and fingerprints computed by this notebook are local outputs and must remain private. The dated-link mode needs PERMNO, CIK, link_start and link_end from an independently reviewed source.

The frozen public SEC snapshot under resources/ is for legacy diagnostics only and must not replace historical security-to-issuer links.


In [ ]:
import sys, json, hashlib, gc, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

sys.path.insert(0, str(CODE_DIR))
from review_data import prepare_return_calendar, attach_identifiers, attach_latest_filing
from review_portfolios import evaluate_diagnostics, validation_summary

def file_sha256(path):
    h = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

names = ['MonthlyReturn.csv', 'Loughran-McDonald_10X_Summaries_1993-2024.csv', 'cik-cusip.csv']
input_checksums = pd.DataFrame([{'file': name, 'bytes': (DATA_DIR / name).stat().st_size,
                                'sha256': file_sha256(DATA_DIR / name)} for name in names])
display(input_checksums)
input_checksums.to_csv(OUTPUT_DIR / 'input_checksums.csv', index=False)
raw = pd.read_csv(DATA_DIR / names[0])
summaries = pd.read_csv(DATA_DIR / names[1])
snapshot_path = REPO_DIR / 'resources/company_tickers_2026-10-30.json'
print('Frozen SEC snapshot SHA256:', file_sha256(snapshot_path))
print('Python', platform.python_version(), '| pandas', pd.__version__, '| NumPy', np.__version__)
print('Return rows:', len(raw), '| LM summary rows:', len(summaries))
print('Last return record:')
display(raw.tail(1))
if len(raw) == 2**20 - 1:
    print('Check whether this source export reached a worksheet row limit before asserting coverage.')


## Prepare targets and select one whole filing record

Targets come from the full raw return table, before ticker or missing-return filtering. Only the immediately following calendar month can be a target. Missing, nonnumeric and absent outcomes remain unresolved.

Latest filings are selected using their full record and a disclosed one Monday-Friday business-day lag. This is an approximation when acceptance timestamps are missing, not a holiday-aware trading calendar. Filings older than three calendar months are unavailable. Invalid new scores do not inherit an older filing's score. Formation eligibility also requires a known current-month return, valid baseline and expanded scores, and an FFInd value.

The published baseline parentheses are retained for a controlled comparison. The separate course guide proposes `(P-N)/(U+1)`; changing it is a design decision requiring a separate comparison and an explicit policy for months without a positive-score sleeve.


In [ ]:
calendar = prepare_return_calendar(raw)
if IDENTIFIER_MODE == 'verified_links':
    links = pd.read_csv(HISTORICAL_LINKS_FILE)  # Fails clearly if no real input exists.
    identified = attach_identifiers(calendar, mode=IDENTIFIER_MODE, historical_links=links)
else:
    snapshot = pd.DataFrame.from_dict(json.loads(snapshot_path.read_text()), orient='index')
    identified = attach_identifiers(calendar, mode=IDENTIFIER_MODE, sec_mapping=snapshot)

panel = attach_latest_filing(identified, summaries, lag_business_days=1,
                             max_age_months=3, baseline_formula=BASELINE_FORMULA)
del raw, summaries, calendar, identified
gc.collect()
eligible = panel[panel.year.between(1994, 2023) & panel.Baseline_Sentiment.notna()
                 & panel.Expanded_LM_Signal.notna() & panel.FFInd.notna()].copy()
print('Common-universe security-months:', len(eligible))
display(eligible.target_status.value_counts().rename('rows').to_frame())
print('Maximum usable filing age:', panel.loc[panel.filing_usable, 'filing_age_months'].max())
assert panel.loc[panel.filing_usable, 'filing_age_months'].max() <= 3
assert panel.loc[panel.filing_usable, 'available_date'].le(panel.loc[panel.filing_usable, 'date']).all()
eligible.loc[eligible.Next_Month_Return.isna(),
             ['PERMNO', 'CIK', 'Ticker', 'COMNAM', 'formation_month', 'return_month',
              'target_status', 'target_return_code', 'ACC_NUM']].to_csv(OUTPUT_DIR / 'unresolved_eligible_outcomes.csv', index=False)


## Form holdings, then evaluate their actual payoffs

Baseline uses positive and negative scores, with positive short magnitudes and exactly one subtraction. Expanded uses global quintiles. Enhanced uses within-industry quintiles, matched industry allocations and a hard 5% cap per name relative to capital. Industries with fewer than five names do not participate in the enhanced strategy. Infeasible capacity remains unallocated; the cap is never bypassed. When either baseline sleeve is absent, both sleeves remain unallocated.

Only formation-time signals are winsorized. Realized returns are preserved. Missing outcomes do not alter holdings. A missing held payoff makes that month's portfolio return unavailable. Full-period statistics are blocked while historical identities, portfolio outcomes or benchmark outcomes remain unresolved.


In [ ]:
monthly, unresolved = evaluate_diagnostics(panel)
summary = validation_summary(monthly, identifier_mode=IDENTIFIER_MODE)
monthly.to_csv(OUTPUT_DIR / 'monthly_portfolio_diagnostics.csv', index=False)
unresolved.to_csv(OUTPUT_DIR / 'unresolved_held_outcomes.csv', index=False)
summary.to_csv(OUTPUT_DIR / 'validation_summary.csv', index=False)
display(summary)
print('Unresolved held-position rows across the three strategies:', len(unresolved))
display(unresolved.head(12))
enhanced = monthly[monthly.Strategy.eq('Enhanced')]
assert enhanced.max_security_weight.max() <= 0.05 + 1e-10
assert enhanced.max_absolute_industry_net.max() <= 1e-10
print('Enhanced maximum name weight:', enhanced.max_security_weight.max())
print('Enhanced maximum absolute industry net:', enhanced.max_absolute_industry_net.max())


In [ ]:
counts = summary.set_index('Strategy')[['complete_portfolio_months', 'incomplete_portfolio_months']]
fig, ax = plt.subplots(figsize=(9, 4.2))
counts.plot.barh(stacked=True, color=['#187a75', '#c87738'], ax=ax, legend=False)
ax.set_xlabel('Formation months (360 per strategy)')
ax.set_ylabel('')
ax.set_title('Payoff availability in the review candidate')
ax.legend(['All held outcomes known', 'At least one held outcome unresolved'],
          loc='upper left', bbox_to_anchor=(0, -0.25), ncol=1, frameon=False)
ax.set_xlim(0, 360)
ax.set_xticks([0, 60, 120, 180, 240, 300, 360])
for y, (known, unknown) in enumerate(counts.to_numpy()):
    ax.text(known / 2, y, str(known), ha='center', va='center', color='white')
    ax.text(known + unknown / 2, y, str(unknown), ha='center', va='center', color='white')
fig.subplots_adjust(left=0.15, right=0.97, top=0.87, bottom=0.36)
plt.savefig(OUTPUT_DIR / 'payoff_availability.png', dpi=170, bbox_inches='tight')
plt.show()
print('Known outcomes do not establish correct historical identities or tradability.')
print('No wealth curve is shown while full-period performance is blocked.')


## Descriptive regression sensitivity

These month-fixed-effect regressions use only observed outcomes, and report two-way clustering by issuer (CIK) and month, with small-sample and degrees-of-freedom corrections and a t reference distribution. This complete-case analysis is conditional on the linked, observed sample. It does not resolve missing portfolio payoffs or identity errors. Full-model R-squared includes the month effects; the incremental R-squared isolates the addition of this one signal to the month-only model.


In [ ]:
if RUN_PANEL_REGRESSIONS:
    from review_inference import descriptive_regressions
    inference = descriptive_regressions(panel)
    inference.to_csv(OUTPUT_DIR / 'descriptive_regressions.csv', index=False)
    display(inference)
else:
    print('Descriptive regressions skipped by configuration.')


## Interpretation and next validation steps

The current outputs do not establish market-wide prediction, causality, profitability, underpricing or an out-of-sample edge. A comparison changing accounting, allocation, sample and outcome treatment cannot show that complexity caused worse performance.

Before reporting complete strategy results, obtain a confirmed complete return extract, validate dated issuer links, and resolve missing/terminal payoffs using source documentation and appropriate delisting records. Agree on execution timing, eligibility, capital, financing, borrow and transaction costs, then precommit a holdout or walk-forward protocol.

The original implementation is retained under archive/ as superseded evidence. Its saved positive baseline and figures must not be presented as current validated results. docs/RESULTS.md explains the accounting-only reproduction.
